# 35 - Camera-ready checks (EACL 2027 Findings, commitment #839)

Two experiments promised in the author response and the commitment:

* **Part A - Differential Attention, official head layout.** The paper's implementation kept all
  heads and split each into two half-width maps (20 dims per map at 10M). The official code halves
  the number of heads and keeps each map at the baseline head width (40 dims at 10M). Parameter
  count is unchanged. Rungs 5.25-29.4M, seeds 0 and 1.
* **Part B/C - Learning-rate check.** combo, conv and diff (paper layout) at 10M and 29.4M with
  peak LR 1e-4 and 1e-3 (the paper uses 3e-4). Seed 0 first (Part B), then seed 1 (Part C).

Every run uses the paper's exact protocol, so Delta is seed-paired with the existing runs.


In [1]:
%matplotlib inline
import os, math, time, csv, json, gc
from contextlib import nullcontext
import numpy as np
import torch
from model import Config, GPT, count_params

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "")
D = "../dataset"
train_data = torch.from_numpy(np.load(D + "/wt103_bpe_train.npy"))
val_data = torch.from_numpy(np.load(D + "/wt103_bpe_val.npy"))
vocab = json.load(open(D + "/wt103_bpe_meta.json"))["vocab"]
print(f"vocab={vocab}  train={len(train_data):,}  val={len(val_data):,}")

BLOCK, BATCH = 256, 16
CKPT_DIR = "../results/camready_ckpt"
os.makedirs(CKPT_DIR, exist_ok=True)
# One line per event (start / resume / every checkpoint / finish / divergence). Cell prints reach the
# notebook only when a cell ends, so this file is what the live monitor watches.
PROGRESS_LOG = "../results/camready_progress.log"
def note(msg):
    line = f"{time.strftime('%Y-%m-%d %H:%M:%S')} {msg}"
    print("    " + line, flush=True)
    with open(PROGRESS_LOG, "a") as f:
        f.write(line + "\n")

COMBO = {"pos_encoding": "rope", "activation": "swiglu", "norm_placement": "pre", "norm_type": "rmsnorm"}
SIZES = {"5M": (256, 4, 4), "10M": (320, 6, 8), "15M": (384, 7, 8), "20M": (448, 7, 8), "30M": (512, 8, 8)}
# "_head_div": divide n_head by this. combo_diff_off = official layout (half the heads, full-width maps)
VARIANTS = {"combo": COMBO,
            "combo_conv3": {**COMBO, "conv_kernel": 3},
            "combo_diff": {**COMBO, "attention": "diff"},
            "combo_diff_off": {**COMBO, "attention": "diff", "_head_div": 2}}
RUNS_CSV = "../results/camready_runs.csv"
HEADER = ["part", "size", "variant", "lr", "seed", "params_M", "val_loss", "val_ppl", "minutes"]
if not os.path.exists(RUNS_CSV):
    with open(RUNS_CSV, "w", newline="") as f:
        csv.writer(f).writerow(HEADER)

def get_batch(split, gen):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - BLOCK, (BATCH,), generator=gen)
    x = torch.stack([d[i:i + BLOCK] for i in ix]); y = torch.stack([d[i + 1:i + 1 + BLOCK] for i in ix])
    return x.to(device), y.to(device)
def autocast_ctx():
    return torch.autocast(device_type="cuda", dtype=torch.bfloat16) if device == "cuda" else nullcontext()

def make_cfg(size, variant):
    ne, nl, nh = SIZES[size]
    over = dict(VARIANTS[variant]); div = over.pop("_head_div", 1)
    return Config(vocab_size=vocab, block_size=BLOCK, n_layer=nl, n_head=nh // div, n_embd=ne,
                  dropout=0.1, **over)

# --- power-cut-safe checkpoints -------------------------------------------------------------------
# Written to .tmp, forced to disk (fsync), then swapped in; the previous checkpoint is kept as .prev.
# A file cut short by a power failure fails to load, and loading falls back to the older one.
def ckpt_file(tag, seed):
    return f"{CKPT_DIR}/{tag}_s{seed}.pt"

def save_durable(obj, path):
    tmp = path + ".tmp"
    with open(tmp, "wb") as f:
        torch.save(obj, f); f.flush(); os.fsync(f.fileno())
    if os.path.exists(path): os.replace(path, path + ".prev")
    os.replace(tmp, path)

def load_latest(path):
    for p in (path, path + ".prev"):
        if os.path.exists(p):
            try:
                return torch.load(p, map_location="cpu"), p   # CPU: Generator.set_state needs a CPU ByteTensor
            except Exception as e:
                note(f"WARNING unreadable checkpoint {os.path.basename(p)} ({type(e).__name__}); trying the older one")
    return None, None

def remove_ckpts(tag, seed):
    for q in (ckpt_file(tag, seed), ckpt_file(tag, seed) + ".prev", ckpt_file(tag, seed) + ".tmp"):
        if os.path.exists(q): os.remove(q)

def train_one(tag, size, variant, seed, lr, max_iters=6000, eval_iters=200, warmup=200, ckpt_every=500):
    min_lr = lr / 10.0
    ckpt_path = ckpt_file(tag, seed)
    torch.manual_seed(seed)
    cfg = make_cfg(size, variant)
    model = GPT(cfg).to(device)
    pm = count_params(model) / 1e6
    opt = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.99), weight_decay=0.1)
    tg = torch.Generator().manual_seed(7777 + seed)
    start_it, prior = 0, 0.0   # prior = training seconds spent before an interruption
    ck, src = load_latest(ckpt_path)
    if ck is not None:
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
        tg.set_state(ck["tg"].cpu()); start_it = ck["iter"]; prior = ck.get("elapsed", 0.0)
        # dropout RNG, so a resumed run continues exactly (older checkpoints lack these keys)
        if "rng_cpu" in ck: torch.set_rng_state(ck["rng_cpu"].cpu())
        if "rng_cuda" in ck and device == "cuda": torch.cuda.set_rng_state(ck["rng_cuda"].cpu())
        note(f"[{tag} s{seed}] RESUMED from iter {start_it} ({os.path.basename(src)})")
    else:
        note(f"[{tag} s{seed}] START ({pm:.3f}M params)")
    t0 = time.time()
    def get_lr(it):
        if it < warmup: return lr * (it + 1) / warmup
        r = (it - warmup) / max(1, max_iters - warmup)
        return min_lr + 0.5 * (lr - min_lr) * (1 + math.cos(math.pi * r))
    def save_ckpt(it):
        st = {"model": model.state_dict(), "opt": opt.state_dict(), "iter": it, "tg": tg.get_state(),
              "rng_cpu": torch.get_rng_state(), "elapsed": prior + time.time() - t0}
        if device == "cuda": st["rng_cuda"] = torch.cuda.get_rng_state()
        save_durable(st, ckpt_path)
    @torch.no_grad()
    def ev():
        model.eval(); g = torch.Generator().manual_seed(4242); tot = 0.0
        for _ in range(eval_iters):
            x, y = get_batch("val", g)
            with autocast_ctx():
                _, loss = model(x, y)
            tot += loss.item()
        model.train(); return tot / eval_iters
    diverged = False
    for it in range(start_it, max_iters):
        for grp in opt.param_groups: grp["lr"] = get_lr(it)
        x, y = get_batch("train", tg)
        with autocast_ctx():
            _, loss = model(x, y)
        opt.zero_grad(set_to_none=True); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0); opt.step()
        if (it + 1) % 100 == 0:
            lv = loss.item()
            if not math.isfinite(lv):
                note(f"[{tag} s{seed}] DIVERGED at iter {it + 1} (loss {lv})")
                diverged = True; break
        if (it + 1) % ckpt_every == 0:
            save_ckpt(it + 1)
            note(f"[{tag} s{seed}] iter {it + 1}/{max_iters}  train_loss {loss.item():.3f}  (checkpoint saved)")
    vl = float("nan") if diverged else ev()
    ppl = math.exp(vl) if math.isfinite(vl) and vl < 50 else float("nan")
    mins = (prior + time.time() - t0) / 60   # total training time, including before an interruption
    del model, opt; gc.collect(); torch.cuda.empty_cache()
    return vl, ppl, pm, mins   # the checkpoint is deleted by run() only after the result is on disk

def done_keys():
    with open(RUNS_CSV) as f:
        r = csv.reader(f); next(r, None)
        return {tuple(row[:5]) for row in r if len(row) == len(HEADER)}

def append_row(row):
    # if a power failure cut the last row short, start a new line so the rows cannot merge
    with open(RUNS_CSV, "rb") as f:
        last = b"\n"
        if f.seek(0, 2) > 0:
            f.seek(-1, 2); last = f.read(1)
    with open(RUNS_CSV, "a", newline="") as f:
        if last != b"\n": f.write("\n")
        csv.writer(f).writerow(row); f.flush(); os.fsync(f.fileno())

def run(part, size, variant, lr, seeds):
    t_cell = time.time()
    for s in seeds:
        key = (part, size, variant, f"{lr:g}", str(s))
        tag = f"{part}_{size}_{variant}_lr{lr:g}"
        if key in done_keys():
            remove_ckpts(tag, s)   # left over if the power failed between saving the result and cleanup
            print(f"  {part} {size} {variant} lr={lr:g} s{s}: done, skip", flush=True); continue
        vl, p, pm, mins = train_one(tag, size, variant, s, lr)
        append_row([part, size, variant, f"{lr:g}", s, f"{pm:.3f}", f"{vl:.5f}", f"{p:.4f}", f"{mins:.1f}"])
        remove_ckpts(tag, s)
        note(f"DONE {part} {size} {variant} lr={lr:g} seed {s}: ppl {p:.3f} ({pm:.3f}M) | {mins:.1f} min")
    print(f"cell done in {(time.time() - t_cell) / 60:.1f} min", flush=True)

# parameter check before any training: the official layout must match the paper's diff count
for sz in SIZES:
    pc = {v: count_params(GPT(make_cfg(sz, v))) / 1e6 for v in ("combo", "combo_diff", "combo_diff_off")}
    c = make_cfg(sz, "combo_diff"); o = make_cfg(sz, "combo_diff_off")
    print(f"{sz:4s} params combo {pc['combo']:.3f}M  diff {pc['combo_diff']:.3f}M  diff_off {pc['combo_diff_off']:.3f}M"
          f" | per-map dim: paper {c.n_embd // c.n_head // 2}, official {o.n_embd // o.n_head // 2}"
          f" (baseline head dim {c.n_embd // c.n_head})")
print("ready")


device: cuda NVIDIA GeForce RTX 3070
vocab=8192  train=14,923,628  val=281,169


5M   params combo 5.254M  diff 5.254M  diff_off 5.255M | per-map dim: paper 32, official 64 (baseline head dim 64)


10M  params combo 10.016M  diff 10.017M  diff_off 10.018M | per-map dim: paper 20, official 40 (baseline head dim 40)


15M  params combo 15.566M  diff 15.567M  diff_off 15.568M | per-map dim: paper 24, official 48 (baseline head dim 48)


20M  params combo 20.562M  diff 20.563M  diff_off 20.564M | per-map dim: paper 28, official 56 (baseline head dim 56)


30M  params combo 29.407M  diff 29.409M  diff_off 29.410M | per-map dim: paper 32, official 64 (baseline head dim 64)
ready


In [2]:
run("A", "5M", "combo_diff_off", 3e-4, [0, 1])

<repo>\.venv\Lib\site-packages\torch\nn\modules\module.py:1329: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  return t.to(


    [A_5M_combo_diff_off_lr0.0003 s0] iter 1000/6000  train_loss 5.728


    [A_5M_combo_diff_off_lr0.0003 s0] iter 2000/6000  train_loss 5.328


    [A_5M_combo_diff_off_lr0.0003 s0] iter 3000/6000  train_loss 4.832


    [A_5M_combo_diff_off_lr0.0003 s0] iter 4000/6000  train_loss 4.695


    [A_5M_combo_diff_off_lr0.0003 s0] iter 5000/6000  train_loss 4.587


... (6 outputs left out)

    [A_5M_combo_diff_off_lr0.0003 s1] iter 5000/6000  train_loss 4.549


    [A_5M_combo_diff_off_lr0.0003 s1] iter 6000/6000  train_loss 4.550


  A 5M   combo_diff_off  lr=0.0003 seed 1: ppl 88.463 (5.255M) | 5.0 min


cell done in 10.2 min


In [3]:
run("A", "10M", "combo_diff_off", 3e-4, [0, 1])

    [A_10M_combo_diff_off_lr0.0003 s0] iter 1000/6000  train_loss 5.520


    [A_10M_combo_diff_off_lr0.0003 s0] iter 2000/6000  train_loss 5.160


    [A_10M_combo_diff_off_lr0.0003 s0] iter 3000/6000  train_loss 4.646


    [A_10M_combo_diff_off_lr0.0003 s0] iter 4000/6000  train_loss 4.453


    [A_10M_combo_diff_off_lr0.0003 s0] iter 5000/6000  train_loss 4.366


    [A_10M_combo_diff_off_lr0.0003 s0] iter 6000/6000  train_loss 4.464


... (5 outputs left out)

    [A_10M_combo_diff_off_lr0.0003 s1] iter 5000/6000  train_loss 4.418


    [A_10M_combo_diff_off_lr0.0003 s1] iter 6000/6000  train_loss 4.367


  A 10M  combo_diff_off  lr=0.0003 seed 1: ppl 76.552 (10.018M) | 6.4 min


cell done in 13.5 min


In [4]:
run("A", "15M", "combo_diff_off", 3e-4, [0, 1])

    [A_15M_combo_diff_off_lr0.0003 s0] iter 1000/6000  train_loss 5.433


    [A_15M_combo_diff_off_lr0.0003 s0] iter 2000/6000  train_loss 5.000


    [A_15M_combo_diff_off_lr0.0003 s0] iter 3000/6000  train_loss 4.491


    [A_15M_combo_diff_off_lr0.0003 s0] iter 4000/6000  train_loss 4.272


    [A_15M_combo_diff_off_lr0.0003 s0] iter 5000/6000  train_loss 4.190


    [A_15M_combo_diff_off_lr0.0003 s0] iter 6000/6000  train_loss 4.279


... (5 outputs left out)

    [A_15M_combo_diff_off_lr0.0003 s1] iter 5000/6000  train_loss 4.242


    [A_15M_combo_diff_off_lr0.0003 s1] iter 6000/6000  train_loss 4.173


  A 15M  combo_diff_off  lr=0.0003 seed 1: ppl 65.639 (15.568M) | 7.8 min


cell done in 15.8 min


In [5]:
run("A", "20M", "combo_diff_off", 3e-4, [0, 1])

    [A_20M_combo_diff_off_lr0.0003 s0] iter 1000/6000  train_loss 5.338


    [A_20M_combo_diff_off_lr0.0003 s0] iter 2000/6000  train_loss 4.922


    [A_20M_combo_diff_off_lr0.0003 s0] iter 3000/6000  train_loss 4.397


    [A_20M_combo_diff_off_lr0.0003 s0] iter 4000/6000  train_loss 4.132


    [A_20M_combo_diff_off_lr0.0003 s0] iter 5000/6000  train_loss 4.079


    [A_20M_combo_diff_off_lr0.0003 s0] iter 6000/6000  train_loss 4.158


... (5 outputs left out)

    [A_20M_combo_diff_off_lr0.0003 s1] iter 5000/6000  train_loss 4.171


    [A_20M_combo_diff_off_lr0.0003 s1] iter 6000/6000  train_loss 4.083


  A 20M  combo_diff_off  lr=0.0003 seed 1: ppl 60.966 (20.564M) | 8.6 min


cell done in 17.4 min


In [6]:
run("A", "30M", "combo_diff_off", 3e-4, [0, 1])

    [A_30M_combo_diff_off_lr0.0003 s0] iter 1000/6000  train_loss 5.269


    [A_30M_combo_diff_off_lr0.0003 s0] iter 2000/6000  train_loss 4.809


    [A_30M_combo_diff_off_lr0.0003 s0] iter 3000/6000  train_loss 4.260


    [A_30M_combo_diff_off_lr0.0003 s0] iter 4000/6000  train_loss 4.006


    [A_30M_combo_diff_off_lr0.0003 s0] iter 5000/6000  train_loss 3.942


    [A_30M_combo_diff_off_lr0.0003 s0] iter 6000/6000  train_loss 4.028


... (5 outputs left out)

    [A_30M_combo_diff_off_lr0.0003 s1] iter 5000/6000  train_loss 4.036


    [A_30M_combo_diff_off_lr0.0003 s1] iter 6000/6000  train_loss 3.955


  A 30M  combo_diff_off  lr=0.0003 seed 1: ppl 55.107 (29.410M) | 10.8 min


cell done in 21.6 min


In [7]:
run("B", "10M", "combo", 1e-4, [0])

    [B_10M_combo_lr0.0001 s0] iter 1000/6000  train_loss 5.891


    [B_10M_combo_lr0.0001 s0] iter 2000/6000  train_loss 5.529


    [B_10M_combo_lr0.0001 s0] iter 3000/6000  train_loss 5.131


    [B_10M_combo_lr0.0001 s0] iter 4000/6000  train_loss 4.988


    [B_10M_combo_lr0.0001 s0] iter 5000/6000  train_loss 4.887


    [B_10M_combo_lr0.0001 s0] iter 6000/6000  train_loss 5.001


  B 10M  combo           lr=0.0001 seed 0: ppl 129.954 (10.016M) | 5.1 min


cell done in 5.1 min


In [8]:
run("B", "10M", "combo_conv3", 1e-4, [0])

    [B_10M_combo_conv3_lr0.0001 s0] iter 1000/6000  train_loss 6.035


    [B_10M_combo_conv3_lr0.0001 s0] iter 2000/6000  train_loss 5.662


    [B_10M_combo_conv3_lr0.0001 s0] iter 3000/6000  train_loss 5.332


    [B_10M_combo_conv3_lr0.0001 s0] iter 4000/6000  train_loss 5.209


    [B_10M_combo_conv3_lr0.0001 s0] iter 5000/6000  train_loss 5.115


    [B_10M_combo_conv3_lr0.0001 s0] iter 6000/6000  train_loss 5.230


  B 10M  combo_conv3     lr=0.0001 seed 0: ppl 168.226 (10.026M) | 5.7 min


cell done in 5.7 min


In [9]:
run("B", "10M", "combo_diff", 1e-4, [0])

    [B_10M_combo_diff_lr0.0001 s0] iter 1000/6000  train_loss 6.046


    [B_10M_combo_diff_lr0.0001 s0] iter 2000/6000  train_loss 5.698


    [B_10M_combo_diff_lr0.0001 s0] iter 3000/6000  train_loss 5.323


    [B_10M_combo_diff_lr0.0001 s0] iter 4000/6000  train_loss 5.225


    [B_10M_combo_diff_lr0.0001 s0] iter 5000/6000  train_loss 5.138


    [B_10M_combo_diff_lr0.0001 s0] iter 6000/6000  train_loss 5.226


  B 10M  combo_diff      lr=0.0001 seed 0: ppl 168.162 (10.017M) | 7.5 min


cell done in 7.5 min


In [10]:
run("B", "10M", "combo", 1e-3, [0])

    [B_10M_combo_lr0.001 s0] iter 1000/6000  train_loss 5.068


    [B_10M_combo_lr0.001 s0] iter 2000/6000  train_loss 4.734


    [B_10M_combo_lr0.001 s0] iter 3000/6000  train_loss 4.268


    [B_10M_combo_lr0.001 s0] iter 4000/6000  train_loss 4.037


    [B_10M_combo_lr0.001 s0] iter 5000/6000  train_loss 3.965


    [B_10M_combo_lr0.001 s0] iter 6000/6000  train_loss 4.016


  B 10M  combo           lr=0.001 seed 0: ppl 53.845 (10.016M) | 5.0 min


cell done in 5.0 min


In [11]:
run("B", "10M", "combo_conv3", 1e-3, [0])

    [B_10M_combo_conv3_lr0.001 s0] iter 1000/6000  train_loss 4.797


    [B_10M_combo_conv3_lr0.001 s0] iter 2000/6000  train_loss 4.431


    [B_10M_combo_conv3_lr0.001 s0] iter 3000/6000  train_loss 4.054


    [B_10M_combo_conv3_lr0.001 s0] iter 4000/6000  train_loss 3.795


    [B_10M_combo_conv3_lr0.001 s0] iter 5000/6000  train_loss 3.755


    [B_10M_combo_conv3_lr0.001 s0] iter 6000/6000  train_loss 3.848


  B 10M  combo_conv3     lr=0.001 seed 0: ppl 46.824 (10.026M) | 5.6 min


cell done in 5.6 min


In [12]:
run("B", "10M", "combo_diff", 1e-3, [0])

    [B_10M_combo_diff_lr0.001 s0] iter 1000/6000  train_loss 5.148


    [B_10M_combo_diff_lr0.001 s0] iter 2000/6000  train_loss 4.809


    [B_10M_combo_diff_lr0.001 s0] iter 3000/6000  train_loss 4.336


    [B_10M_combo_diff_lr0.001 s0] iter 4000/6000  train_loss 4.070


    [B_10M_combo_diff_lr0.001 s0] iter 5000/6000  train_loss 4.024


    [B_10M_combo_diff_lr0.001 s0] iter 6000/6000  train_loss 4.087


  B 10M  combo_diff      lr=0.001 seed 0: ppl 59.078 (10.017M) | 7.5 min


cell done in 7.5 min


In [13]:
run("B", "30M", "combo", 1e-4, [0])

    [B_30M_combo_lr0.0001 s0] iter 1000/6000  train_loss 5.701


    [B_30M_combo_lr0.0001 s0] iter 2000/6000  train_loss 5.351


    [B_30M_combo_lr0.0001 s0] iter 3000/6000  train_loss 4.859


    [B_30M_combo_lr0.0001 s0] iter 4000/6000  train_loss 4.683


    [B_30M_combo_lr0.0001 s0] iter 5000/6000  train_loss 4.601


    [B_30M_combo_lr0.0001 s0] iter 6000/6000  train_loss 4.704


  B 30M  combo           lr=0.0001 seed 0: ppl 99.365 (29.407M) | 9.2 min


cell done in 9.2 min


In [14]:
run("B", "30M", "combo_conv3", 1e-4, [0])

    [B_30M_combo_conv3_lr0.0001 s0] iter 1000/6000  train_loss 5.693


    [B_30M_combo_conv3_lr0.0001 s0] iter 2000/6000  train_loss 5.315


    [B_30M_combo_conv3_lr0.0001 s0] iter 3000/6000  train_loss 4.868


    [B_30M_combo_conv3_lr0.0001 s0] iter 4000/6000  train_loss 4.676


    [B_30M_combo_conv3_lr0.0001 s0] iter 5000/6000  train_loss 4.612


    [B_30M_combo_conv3_lr0.0001 s0] iter 6000/6000  train_loss 4.705


  B 30M  combo_conv3     lr=0.0001 seed 0: ppl 102.774 (29.428M) | 10.3 min


cell done in 10.3 min


In [2]:
run("B", "30M", "combo_diff", 1e-4, [0])

<repo>\.venv\Lib\site-packages\torch\nn\modules\module.py:1329: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  return t.to(


    [B_30M_combo_diff_lr0.0001 s0] RESUMED from 5000


    [B_30M_combo_diff_lr0.0001 s0] iter 6000/6000  train_loss 4.857


  B 30M  combo_diff      lr=0.0001 seed 0: ppl 112.156 (29.409M) | 2.3 min


cell done in 2.3 min


In [2]:
run("B", "30M", "combo", 1e-3, [0])

<repo>\.venv\Lib\site-packages\torch\nn\modules\module.py:1329: UserWarning: expandable_segments not supported on this platform (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\c10/cuda/CUDAAllocatorConfig.h:28.)
  return t.to(


    2026-11-19 09:40:17 [B_30M_combo_lr0.001 s0] START (29.407M params)


    2026-11-19 09:41:03 [B_30M_combo_lr0.001 s0] iter 500/6000  train_loss 5.576  (checkpoint saved)


    2026-11-19 09:41:50 [B_30M_combo_lr0.001 s0] iter 1000/6000  train_loss 5.129  (checkpoint saved)


    2026-11-19 09:42:36 [B_30M_combo_lr0.001 s0] iter 1500/6000  train_loss 4.990  (checkpoint saved)


    2026-11-19 09:43:23 [B_30M_combo_lr0.001 s0] iter 2000/6000  train_loss 4.721  (checkpoint saved)


... (6 outputs left out)

    2026-11-19 09:48:49 [B_30M_combo_lr0.001 s0] iter 5500/6000  train_loss 3.679  (checkpoint saved)


    2026-11-19 09:49:35 [B_30M_combo_lr0.001 s0] iter 6000/6000  train_loss 3.854  (checkpoint saved)


    2026-11-19 09:49:41 DONE B 30M combo lr=0.001 seed 0: ppl 48.279 (29.407M) | 9.4 min


cell done in 9.4 min


In [3]:
run("B", "30M", "combo_conv3", 1e-3, [0])

    2026-11-19 09:49:41 [B_30M_combo_conv3_lr0.001 s0] START (29.428M params)


    2026-11-19 09:50:33 [B_30M_combo_conv3_lr0.001 s0] iter 500/6000  train_loss 5.143  (checkpoint saved)


    2026-11-19 09:51:25 [B_30M_combo_conv3_lr0.001 s0] iter 1000/6000  train_loss 4.633  (checkpoint saved)


    2026-11-19 09:52:17 [B_30M_combo_conv3_lr0.001 s0] iter 1500/6000  train_loss 4.542  (checkpoint saved)


    2026-11-19 09:53:09 [B_30M_combo_conv3_lr0.001 s0] iter 2000/6000  train_loss 4.240  (checkpoint saved)


    2026-11-19 09:54:01 [B_30M_combo_conv3_lr0.001 s0] iter 2500/6000  train_loss 3.897  (checkpoint saved)


... (5 outputs left out)

    2026-11-19 09:59:12 [B_30M_combo_conv3_lr0.001 s0] iter 5500/6000  train_loss 3.376  (checkpoint saved)


    2026-11-19 10:00:03 [B_30M_combo_conv3_lr0.001 s0] iter 6000/6000  train_loss 3.566  (checkpoint saved)


    2026-11-19 10:00:10 DONE B 30M combo_conv3 lr=0.001 seed 0: ppl 39.456 (29.428M) | 10.5 min


cell done in 10.5 min


In [4]:
run("B", "30M", "combo_diff", 1e-3, [0])

    2026-11-19 10:00:10 [B_30M_combo_diff_lr0.001 s0] START (29.409M params)


    2026-11-19 10:01:14 [B_30M_combo_diff_lr0.001 s0] iter 500/6000  train_loss 5.535  (checkpoint saved)


    2026-11-19 10:02:18 [B_30M_combo_diff_lr0.001 s0] iter 1000/6000  train_loss 5.053  (checkpoint saved)


    2026-11-19 10:03:23 [B_30M_combo_diff_lr0.001 s0] iter 1500/6000  train_loss 4.922  (checkpoint saved)


    2026-11-19 10:04:27 [B_30M_combo_diff_lr0.001 s0] iter 2000/6000  train_loss 4.707  (checkpoint saved)


    2026-11-19 10:05:31 [B_30M_combo_diff_lr0.001 s0] iter 2500/6000  train_loss 4.316  (checkpoint saved)


... (5 outputs left out)

    2026-11-19 10:11:55 [B_30M_combo_diff_lr0.001 s0] iter 5500/6000  train_loss 3.689  (checkpoint saved)


    2026-11-19 10:12:59 [B_30M_combo_diff_lr0.001 s0] iter 6000/6000  train_loss 3.859  (checkpoint saved)


    2026-11-19 10:13:07 DONE B 30M combo_diff lr=0.001 seed 0: ppl 50.691 (29.409M) | 12.9 min


cell done in 13.0 min


In [5]:
run("C", "10M", "combo", 1e-4, [1])

    2026-11-19 10:13:07 [C_10M_combo_lr0.0001 s1] START (10.016M params)


    2026-11-19 10:13:32 [C_10M_combo_lr0.0001 s1] iter 500/6000  train_loss 6.331  (checkpoint saved)


    2026-11-19 10:13:58 [C_10M_combo_lr0.0001 s1] iter 1000/6000  train_loss 5.875  (checkpoint saved)


    2026-11-19 10:14:23 [C_10M_combo_lr0.0001 s1] iter 1500/6000  train_loss 5.664  (checkpoint saved)


    2026-11-19 10:14:48 [C_10M_combo_lr0.0001 s1] iter 2000/6000  train_loss 5.541  (checkpoint saved)


    2026-11-19 10:15:13 [C_10M_combo_lr0.0001 s1] iter 2500/6000  train_loss 5.282  (checkpoint saved)


... (5 outputs left out)

    2026-11-19 10:17:45 [C_10M_combo_lr0.0001 s1] iter 5500/6000  train_loss 5.065  (checkpoint saved)


    2026-11-19 10:18:10 [C_10M_combo_lr0.0001 s1] iter 6000/6000  train_loss 4.917  (checkpoint saved)


    2026-11-19 10:18:13 DONE C 10M combo lr=0.0001 seed 1: ppl 130.824 (10.016M) | 5.1 min


cell done in 5.1 min


In [6]:
run("C", "10M", "combo_conv3", 1e-4, [1])

    2026-11-19 10:18:14 [C_10M_combo_conv3_lr0.0001 s1] START (10.026M params)


    2026-11-19 10:18:42 [C_10M_combo_conv3_lr0.0001 s1] iter 500/6000  train_loss 6.571  (checkpoint saved)


    2026-11-19 10:19:10 [C_10M_combo_conv3_lr0.0001 s1] iter 1000/6000  train_loss 6.012  (checkpoint saved)


    2026-11-19 10:19:38 [C_10M_combo_conv3_lr0.0001 s1] iter 1500/6000  train_loss 5.767  (checkpoint saved)


    2026-11-19 10:20:06 [C_10M_combo_conv3_lr0.0001 s1] iter 2000/6000  train_loss 5.672  (checkpoint saved)


    2026-11-19 10:20:34 [C_10M_combo_conv3_lr0.0001 s1] iter 2500/6000  train_loss 5.434  (checkpoint saved)


... (5 outputs left out)

    2026-11-19 10:23:22 [C_10M_combo_conv3_lr0.0001 s1] iter 5500/6000  train_loss 5.294  (checkpoint saved)


    2026-11-19 10:23:50 [C_10M_combo_conv3_lr0.0001 s1] iter 6000/6000  train_loss 5.126  (checkpoint saved)


    2026-11-19 10:23:54 DONE C 10M combo_conv3 lr=0.0001 seed 1: ppl 167.500 (10.026M) | 5.7 min


cell done in 5.7 min


In [7]:
run("C", "10M", "combo_diff", 1e-4, [1])

    2026-11-19 10:23:54 [C_10M_combo_diff_lr0.0001 s1] START (10.017M params)


    2026-11-19 10:24:31 [C_10M_combo_diff_lr0.0001 s1] iter 500/6000  train_loss 6.644  (checkpoint saved)


    2026-11-19 10:25:09 [C_10M_combo_diff_lr0.0001 s1] iter 1000/6000  train_loss 6.055  (checkpoint saved)


    2026-11-19 10:25:47 [C_10M_combo_diff_lr0.0001 s1] iter 1500/6000  train_loss 5.796  (checkpoint saved)


    2026-11-19 10:26:24 [C_10M_combo_diff_lr0.0001 s1] iter 2000/6000  train_loss 5.697  (checkpoint saved)


    2026-11-19 10:27:02 [C_10M_combo_diff_lr0.0001 s1] iter 2500/6000  train_loss 5.471  (checkpoint saved)


... (5 outputs left out)

    2026-11-19 10:30:48 [C_10M_combo_diff_lr0.0001 s1] iter 5500/6000  train_loss 5.322  (checkpoint saved)


    2026-11-19 10:31:25 [C_10M_combo_diff_lr0.0001 s1] iter 6000/6000  train_loss 5.179  (checkpoint saved)


    2026-11-19 10:31:30 DONE C 10M combo_diff lr=0.0001 seed 1: ppl 170.126 (10.017M) | 7.6 min


cell done in 7.6 min


In [8]:
run("C", "10M", "combo", 1e-3, [1])

    2026-11-19 10:31:30 [C_10M_combo_lr0.001 s1] START (10.016M params)


    2026-11-19 10:31:56 [C_10M_combo_lr0.001 s1] iter 500/6000  train_loss 5.374  (checkpoint saved)


    2026-11-19 10:32:21 [C_10M_combo_lr0.001 s1] iter 1000/6000  train_loss 5.019  (checkpoint saved)


    2026-11-19 10:32:46 [C_10M_combo_lr0.001 s1] iter 1500/6000  train_loss 4.811  (checkpoint saved)


    2026-11-19 10:33:11 [C_10M_combo_lr0.001 s1] iter 2000/6000  train_loss 4.643  (checkpoint saved)


    2026-11-19 10:33:37 [C_10M_combo_lr0.001 s1] iter 2500/6000  train_loss 4.281  (checkpoint saved)


... (5 outputs left out)

    2026-11-19 10:36:08 [C_10M_combo_lr0.001 s1] iter 5500/6000  train_loss 3.997  (checkpoint saved)


    2026-11-19 10:36:33 [C_10M_combo_lr0.001 s1] iter 6000/6000  train_loss 3.914  (checkpoint saved)


    2026-11-19 10:36:37 DONE C 10M combo lr=0.001 seed 1: ppl 52.696 (10.016M) | 5.1 min


cell done in 5.1 min


In [9]:
run("C", "10M", "combo_conv3", 1e-3, [1])

    2026-11-19 10:36:37 [C_10M_combo_conv3_lr0.001 s1] START (10.026M params)


    2026-11-19 10:37:05 [C_10M_combo_conv3_lr0.001 s1] iter 500/6000  train_loss 5.191  (checkpoint saved)


    2026-11-19 10:37:33 [C_10M_combo_conv3_lr0.001 s1] iter 1000/6000  train_loss 4.759  (checkpoint saved)


    2026-11-19 10:38:01 [C_10M_combo_conv3_lr0.001 s1] iter 1500/6000  train_loss 4.577  (checkpoint saved)


    2026-11-19 10:38:29 [C_10M_combo_conv3_lr0.001 s1] iter 2000/6000  train_loss 4.388  (checkpoint saved)


    2026-11-19 10:38:57 [C_10M_combo_conv3_lr0.001 s1] iter 2500/6000  train_loss 4.046  (checkpoint saved)


... (5 outputs left out)

    2026-11-19 10:41:45 [C_10M_combo_conv3_lr0.001 s1] iter 5500/6000  train_loss 3.810  (checkpoint saved)


    2026-11-19 10:42:13 [C_10M_combo_conv3_lr0.001 s1] iter 6000/6000  train_loss 3.757  (checkpoint saved)


    2026-11-19 10:42:17 DONE C 10M combo_conv3 lr=0.001 seed 1: ppl 46.705 (10.026M) | 5.7 min


cell done in 5.7 min


In [10]:
run("C", "10M", "combo_diff", 1e-3, [1])

    2026-11-19 10:42:17 [C_10M_combo_diff_lr0.001 s1] START (10.017M params)


    2026-11-19 10:42:55 [C_10M_combo_diff_lr0.001 s1] iter 500/6000  train_loss 5.500  (checkpoint saved)


    2026-11-19 10:43:32 [C_10M_combo_diff_lr0.001 s1] iter 1000/6000  train_loss 5.138  (checkpoint saved)


    2026-11-19 10:44:10 [C_10M_combo_diff_lr0.001 s1] iter 1500/6000  train_loss 4.929  (checkpoint saved)


    2026-11-19 10:44:47 [C_10M_combo_diff_lr0.001 s1] iter 2000/6000  train_loss 4.767  (checkpoint saved)


    2026-11-19 10:45:25 [C_10M_combo_diff_lr0.001 s1] iter 2500/6000  train_loss 4.446  (checkpoint saved)


... (5 outputs left out)

    2026-11-19 10:49:10 [C_10M_combo_diff_lr0.001 s1] iter 5500/6000  train_loss 4.114  (checkpoint saved)


    2026-11-19 10:49:48 [C_10M_combo_diff_lr0.001 s1] iter 6000/6000  train_loss 4.050  (checkpoint saved)


    2026-11-19 10:49:53 DONE C 10M combo_diff lr=0.001 seed 1: ppl 60.504 (10.017M) | 7.6 min


cell done in 7.6 min


In [11]:
run("C", "30M", "combo", 1e-4, [1])

    2026-11-19 10:49:53 [C_30M_combo_lr0.0001 s1] START (29.407M params)


    2026-11-19 10:50:40 [C_30M_combo_lr0.0001 s1] iter 500/6000  train_loss 6.088  (checkpoint saved)


    2026-11-19 10:51:26 [C_30M_combo_lr0.0001 s1] iter 1000/6000  train_loss 5.689  (checkpoint saved)


    2026-11-19 10:52:12 [C_30M_combo_lr0.0001 s1] iter 1500/6000  train_loss 5.476  (checkpoint saved)


    2026-11-19 10:52:58 [C_30M_combo_lr0.0001 s1] iter 2000/6000  train_loss 5.313  (checkpoint saved)


    2026-11-19 10:53:44 [C_30M_combo_lr0.0001 s1] iter 2500/6000  train_loss 5.028  (checkpoint saved)


... (5 outputs left out)

    2026-11-19 10:58:22 [C_30M_combo_lr0.0001 s1] iter 5500/6000  train_loss 4.767  (checkpoint saved)


    2026-11-19 10:59:08 [C_30M_combo_lr0.0001 s1] iter 6000/6000  train_loss 4.638  (checkpoint saved)


    2026-11-19 10:59:14 DONE C 30M combo lr=0.0001 seed 1: ppl 99.567 (29.407M) | 9.3 min


cell done in 9.3 min


In [12]:
run("C", "30M", "combo_conv3", 1e-4, [1])

    2026-11-19 10:59:14 [C_30M_combo_conv3_lr0.0001 s1] START (29.428M params)


    2026-11-19 11:00:06 [C_30M_combo_conv3_lr0.0001 s1] iter 500/6000  train_loss 6.104  (checkpoint saved)


    2026-11-19 11:00:57 [C_30M_combo_conv3_lr0.0001 s1] iter 1000/6000  train_loss 5.686  (checkpoint saved)


    2026-11-19 11:01:49 [C_30M_combo_conv3_lr0.0001 s1] iter 1500/6000  train_loss 5.473  (checkpoint saved)


    2026-11-19 11:02:40 [C_30M_combo_conv3_lr0.0001 s1] iter 2000/6000  train_loss 5.295  (checkpoint saved)


    2026-11-19 11:03:32 [C_30M_combo_conv3_lr0.0001 s1] iter 2500/6000  train_loss 5.002  (checkpoint saved)


... (5 outputs left out)

    2026-11-19 11:08:42 [C_30M_combo_conv3_lr0.0001 s1] iter 5500/6000  train_loss 4.754  (checkpoint saved)


    2026-11-19 11:09:34 [C_30M_combo_conv3_lr0.0001 s1] iter 6000/6000  train_loss 4.635  (checkpoint saved)


    2026-11-19 11:09:40 DONE C 30M combo_conv3 lr=0.0001 seed 1: ppl 103.437 (29.428M) | 10.4 min


cell done in 10.4 min


In [13]:
run("C", "30M", "combo_diff", 1e-4, [1])

    2026-11-19 11:09:40 [C_30M_combo_diff_lr0.0001 s1] START (29.409M params)


    2026-11-19 11:10:44 [C_30M_combo_diff_lr0.0001 s1] iter 500/6000  train_loss 6.165  (checkpoint saved)


    2026-11-19 11:11:47 [C_30M_combo_diff_lr0.0001 s1] iter 1000/6000  train_loss 5.730  (checkpoint saved)


    2026-11-19 11:12:51 [C_30M_combo_diff_lr0.0001 s1] iter 1500/6000  train_loss 5.531  (checkpoint saved)


    2026-11-19 11:13:54 [C_30M_combo_diff_lr0.0001 s1] iter 2000/6000  train_loss 5.409  (checkpoint saved)


    2026-11-19 11:14:57 [C_30M_combo_diff_lr0.0001 s1] iter 2500/6000  train_loss 5.107  (checkpoint saved)


... (5 outputs left out)

    2026-11-19 11:21:17 [C_30M_combo_diff_lr0.0001 s1] iter 5500/6000  train_loss 4.860  (checkpoint saved)


    2026-11-19 11:22:20 [C_30M_combo_diff_lr0.0001 s1] iter 6000/6000  train_loss 4.763  (checkpoint saved)


    2026-11-19 11:22:28 DONE C 30M combo_diff lr=0.0001 seed 1: ppl 112.192 (29.409M) | 12.8 min


cell done in 12.8 min


In [14]:
run("C", "30M", "combo", 1e-3, [1])

    2026-11-19 11:22:29 [C_30M_combo_lr0.001 s1] START (29.407M params)


    2026-11-19 11:23:14 [C_30M_combo_lr0.001 s1] iter 500/6000  train_loss 5.438  (checkpoint saved)


    2026-11-19 11:24:00 [C_30M_combo_lr0.001 s1] iter 1000/6000  train_loss 5.056  (checkpoint saved)


    2026-11-19 11:24:46 [C_30M_combo_lr0.001 s1] iter 1500/6000  train_loss 4.852  (checkpoint saved)


    2026-11-19 11:25:32 [C_30M_combo_lr0.001 s1] iter 2000/6000  train_loss 4.671  (checkpoint saved)


    2026-11-19 11:26:18 [C_30M_combo_lr0.001 s1] iter 2500/6000  train_loss 4.316  (checkpoint saved)


... (5 outputs left out)

    2026-11-19 11:30:52 [C_30M_combo_lr0.001 s1] iter 5500/6000  train_loss 3.863  (checkpoint saved)


    2026-11-19 11:31:38 [C_30M_combo_lr0.001 s1] iter 6000/6000  train_loss 3.769  (checkpoint saved)


    2026-11-19 11:31:44 DONE C 30M combo lr=0.001 seed 1: ppl 48.924 (29.407M) | 9.3 min


cell done in 9.3 min


In [15]:
run("C", "30M", "combo_conv3", 1e-3, [1])

    2026-11-19 11:31:44 [C_30M_combo_conv3_lr0.001 s1] START (29.428M params)


    2026-11-19 11:32:36 [C_30M_combo_conv3_lr0.001 s1] iter 500/6000  train_loss 5.038  (checkpoint saved)


    2026-11-19 11:33:27 [C_30M_combo_conv3_lr0.001 s1] iter 1000/6000  train_loss 4.594  (checkpoint saved)


    2026-11-19 11:34:18 [C_30M_combo_conv3_lr0.001 s1] iter 1500/6000  train_loss 4.383  (checkpoint saved)


    2026-11-19 11:35:09 [C_30M_combo_conv3_lr0.001 s1] iter 2000/6000  train_loss 4.230  (checkpoint saved)


    2026-11-19 11:36:00 [C_30M_combo_conv3_lr0.001 s1] iter 2500/6000  train_loss 3.853  (checkpoint saved)


... (5 outputs left out)

    2026-11-19 11:41:08 [C_30M_combo_conv3_lr0.001 s1] iter 5500/6000  train_loss 3.491  (checkpoint saved)


    2026-11-19 11:41:59 [C_30M_combo_conv3_lr0.001 s1] iter 6000/6000  train_loss 3.468  (checkpoint saved)


    2026-11-19 11:42:05 DONE C 30M combo_conv3 lr=0.001 seed 1: ppl 39.629 (29.428M) | 10.3 min


cell done in 10.4 min


In [16]:
run("C", "30M", "combo_diff", 1e-3, [1])

    2026-11-19 11:42:06 [C_30M_combo_diff_lr0.001 s1] START (29.409M params)


    2026-11-19 11:43:09 [C_30M_combo_diff_lr0.001 s1] iter 500/6000  train_loss 5.382  (checkpoint saved)


    2026-11-19 11:44:12 [C_30M_combo_diff_lr0.001 s1] iter 1000/6000  train_loss 5.028  (checkpoint saved)


    2026-11-19 11:45:16 [C_30M_combo_diff_lr0.001 s1] iter 1500/6000  train_loss 4.797  (checkpoint saved)


    2026-11-19 11:46:19 [C_30M_combo_diff_lr0.001 s1] iter 2000/6000  train_loss 4.673  (checkpoint saved)


    2026-11-19 11:47:22 [C_30M_combo_diff_lr0.001 s1] iter 2500/6000  train_loss 4.279  (checkpoint saved)


... (5 outputs left out)

    2026-11-19 11:53:42 [C_30M_combo_diff_lr0.001 s1] iter 5500/6000  train_loss 3.865  (checkpoint saved)


    2026-11-19 11:54:45 [C_30M_combo_diff_lr0.001 s1] iter 6000/6000  train_loss 3.772  (checkpoint saved)


    2026-11-19 11:54:53 DONE C 30M combo_diff lr=0.001 seed 1: ppl 50.541 (29.409M) | 12.8 min


cell done in 12.8 min


In [17]:
# === Summary: compare with the paper's runs at LR 3e-4 (seed-paired) ===
paper = {}   # (size, variant, seed) -> ppl at LR 3e-4
with open("../results/ladder6000_runs.csv") as f:
    r = csv.reader(f); next(r, None)
    for sz, v, s, pm, pp in r: paper[(sz, v, s)] = float(pp)
for fn, s in (("../results/scale_confirm_runs.csv", "0"), ("../results/scale_confirm_s1_runs.csv", "1")):
    with open(fn) as f:
        r = csv.reader(f); next(r, None)
        for row in r:
            if len(row) == 3: paper[("30M", row[0], s)] = float(row[2])
new = {}
with open(RUNS_CSV) as f:
    r = csv.DictReader(f)
    for row in r: new[(row["size"], row["variant"], row["lr"], row["seed"])] = float(row["val_ppl"])
def dlt(v, c): return 100.0 * (v - c) / c

print("=== Part A: Differential Attention, paper layout vs official layout (Delta % vs combo, LR 3e-4) ===")
print(f"{'size':5s} {'seed':4s} {'combo':>8s} {'paper':>8s} {'official':>9s}")
for sz in SIZES:
    rows = []
    for s in ("0", "1"):
        c = paper.get((sz, "combo", s)); dp = paper.get((sz, "combo_diff", s)); do = new.get((sz, "combo_diff_off", "0.0003", s))
        if c and dp and do:
            rows.append((dlt(dp, c), dlt(do, c)))
            print(f"{sz:5s} {s:4s} {c:8.2f} {dlt(dp, c):+8.2f} {dlt(do, c):+9.2f}")
    if rows:
        print(f"{sz:5s} mean {'':8s} {np.mean([a for a, b in rows]):+8.2f} {np.mean([b for a, b in rows]):+9.2f}")

print("\n=== Parts B/C: learning-rate check (Delta % vs combo at the same LR and seed) ===")
for sz in ("10M", "30M"):
    for s in ("0", "1"):
        line = f"{sz:4s} s{s}: "
        for lr in ("0.0001", "0.0003", "0.001"):
            if lr == "0.0003":
                c = paper.get((sz, "combo", s)); cv = paper.get((sz, "combo_conv3", s)); df = paper.get((sz, "combo_diff", s))
            else:
                c = new.get((sz, "combo", lr, s)); cv = new.get((sz, "combo_conv3", lr, s)); df = new.get((sz, "combo_diff", lr, s))
            if c and cv and df and all(map(math.isfinite, (c, cv, df))):
                line += f" | lr {float(lr):g}: combo {c:6.2f} conv {dlt(cv, c):+6.2f} diff {dlt(df, c):+6.2f}"
            else:
                line += f" | lr {float(lr):g}: (missing/diverged)"
        print(line)


=== Part A: Differential Attention, paper layout vs official layout (Delta % vs combo, LR 3e-4) ===
size  seed    combo    paper  official
5M    0       82.50   +14.67    +14.49
5M    1       83.14   +16.77     +6.40
5M    mean            +15.72    +10.44
10M   0       68.27   +20.23    +12.23
10M   1       67.58   +23.41    +13.28
10M   mean            +21.82    +12.75
15M   0       61.74   +13.98     +7.24
15M   1       64.23   +10.69     +2.19
15M   mean            +12.33     +4.71
20M   0       59.97    +4.30     +0.19
20M   1       60.47    +8.14     +0.82
20M   mean             +6.22     +0.50
30M   0       55.14    +6.19     -1.95
30M   1       55.77    +1.71     -1.20
30M   mean             +3.95     -1.57

=== Parts B/C: learning-rate check (Delta % vs combo at the same LR and seed) ===
10M  s0:  | lr 0.0001: combo 129.95 conv +29.45 diff +29.40 | lr 0.0003: combo  68.27 conv  +3.86 diff +20.23 | lr 0.001: combo  53.84 conv -13.04 diff  +9.72
10M  s1:  | lr 0.0001: combo 130.8